# Climate finance and forest-restoration potential

## Purpose and scope

This deep dive follows two separate calculations. The first distributes the hypothetical 2% GCW allocation across study entities and economic groups. The second reconstructs forest-restoration costs and supply from the public cell-level Busch inputs.

The restoration calculation runs for every country `.dta` file in the archive before any link to FRA losses or to the study entities. A later ISO3 join attaches positive FRA-loss targets to the already calculated country outputs. The 2% allocation is not observed finance or an estimated funding need, and the restoration scenario does not directly price protection of existing forests.

## Load the allocation, all-country restoration outputs and the later target join

The country-cost files describe every `.dta` input processed from the Busch archive. The separate coverage file contains only study entities with positive FRA loss and records whether an ISO3-matched country calculation is available. A country without a positive-loss target remains in the all-country cost files; it is simply absent from that target-coverage table.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "code" / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository from the notebook working directory.")

ROOT = find_repo_root(Path.cwd().resolve())
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Run this notebook from within the repository.')

DATA = ROOT / 'data' / 'derived'
RESULTS = ROOT / 'results'
TABLES = RESULTS / 'tables' / 'manuscript'
VALIDATION = ROOT / 'data' / 'derived' / 'validation'
BUSCH_DIR = DATA / 'busch_reconstruction'
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
print(f'Repository: {ROOT}')

## 2. Examine how the 2% GCW scenario is distributed

The allocation is proportional to the annual GCW value of each eligible study unit, then summarized by economic group. Compare both the total assigned to a group and its share of the global scenario; the median per-unit value describes a different aspect of the distribution.

In [ ]:
allocation = pd.read_csv(TABLES / 'Table4_our_2pct_gcw_allocation.csv')
assumptions = pd.read_csv(TABLES / 'Supplementary_deforestation_scenarios.csv')
accounts = pd.read_csv(TABLES / 'Supplementary_country_level_accounts_2024.csv')

# Reconstruct the stipulated allocation from annual GCW and compare group sums with Table 4.
allocation_rate = 0.02
gcw = pd.to_numeric(accounts['gcw_usd2015_yr_mean'], errors='coerce')
exclude = accounts['exclude_from_country_sum'].fillna(False).astype(bool)
eligible = gcw.notna() & ~exclude
accounts.loc[eligible, 'reconstructed_allocation_usd2015_yr'] = allocation_rate * gcw.loc[eligible]
reconstructed = accounts.loc[eligible].groupby('groups', dropna=False)['reconstructed_allocation_usd2015_yr'].sum().div(1e9).rename('reconstructed_billion_usd2015').reset_index()
comparison = allocation.merge(reconstructed, left_on='group', right_on='groups', how='outer')
comparison['difference_billion_usd2015'] = comparison['reconstructed_billion_usd2015'] - comparison['annual_allocation_billion_usd2015']
display(comparison)
display(assumptions.loc[assumptions['metric'].astype(str).str.contains('rate|allocation|discount', case=False, na=False)])


### Visual summary: annual allocation by economic group

The bars show the hypothetical annual amount assigned under the same 2% rule. They do not show payments made, pledged finance or a recommended contribution.

In [ ]:
allocation_plot = allocation.loc[allocation["group"].ne("Total")].copy()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(allocation_plot["group"], allocation_plot["annual_allocation_billion_usd2015"], color="#2878A5")
ax.set_ylabel("Hypothetical annual allocation (billion 2015 US$/year)")
ax.set_title("2% GCW allocation by economic group")
ax.tick_params(axis="x", rotation=18)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## 3. Examine restoration supply and costs for every archive country

The country-cost output is calculated from each country `.dta` file independently of FRA loss and study-table coverage. Use it to inspect all calculated country supplies and cost curves. The coverage table is created afterward: it joins only positive FRA-loss targets by ISO3 so that those targets can be compared with the previously calculated supply. No published national curve is used to select or validate the calculation.

In [ ]:
import json

BUSCH_DIR = DATA / "busch_reconstruction"
busch_prefix = "busch_all_dta"
busch_summary = json.loads((BUSCH_DIR / f"{busch_prefix}_summary.json").read_text(encoding="utf-8"))
all_country_costs = pd.read_csv(BUSCH_DIR / f"{busch_prefix}_country_finance.csv")
all_country_price_curves = pd.read_csv(BUSCH_DIR / f"{busch_prefix}_25_price_curves.csv")
positive_loss_coverage = pd.read_csv(BUSCH_DIR / f"{busch_prefix}_coverage.csv")

archive_scope = pd.DataFrame([
    {"measure": "Country DTA files in archive", "count": busch_summary["archive_country_dta_count"]},
    {"measure": "Country DTA files processed", "count": busch_summary["processed_country_dta_count"]},
    {"measure": "Study entities with positive FRA loss", "count": busch_summary["positive_loss_entities"]},
    {"measure": "Positive-loss entities with a country DTA", "count": busch_summary["positive_loss_entities_with_dta"]},
    {"measure": "Positive-loss entities without a country DTA", "count": busch_summary["positive_loss_entities_without_dta"]},
])
display(archive_scope)

display(
    all_country_costs.sort_values("available_discounted_tco2", ascending=False)
    [["iso3", "output_cells", "selected_natural_regeneration_cells",
      "selected_plantation_cells", "available_discounted_tco2"]]
    .head(20)
)
display(positive_loss_coverage.head(20))

# Summing the country-level cumulative curves gives an archive-wide descriptive curve.
archive_price_curve = (
    all_country_price_curves.groupby("price_usd2015_per_tco2", as_index=False)
    ["cumulative_supply_discounted_tco2"].sum()
    .sort_values("price_usd2015_per_tco2")
)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    archive_price_curve["price_usd2015_per_tco2"],
    archive_price_curve["cumulative_supply_discounted_tco2"] / 1e6,
    color="#2878A5",
)
ax.set_xlabel("Price threshold (2015 US$/tCO₂)")
ax.set_ylabel("Cumulative modelled supply (MtCO₂, discounted over 30 years)")
ax.set_title("Restoration supply across all country DTA inputs")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## 4. Compare the natural-regeneration soil scenarios

The reference calculation and two soil-rate alternatives are each recalculated for every country `.dta` input. Only after those calculations does a second table join the resulting scenarios to positive FRA-loss targets. The plantation soil rate stays fixed. These are deterministic sensitivity scenarios, not confidence intervals.

In [ ]:
soil_sensitivity_path = BUSCH_DIR / f"{busch_prefix}_soil_sensitivity_country.csv"
sensitivity_curves_path = BUSCH_DIR / f"{busch_prefix}_soil_sensitivity_price_curves.csv"
sensitivity_coverage_path = BUSCH_DIR / f"{busch_prefix}_soil_sensitivity_coverage.csv"

soil_sensitivity = pd.read_csv(soil_sensitivity_path)
sensitivity_summary = (
    soil_sensitivity.groupby("soil_scenario", dropna=False)
    .agg(
        nr_soil_rate_tC_ha_yr=("nr_soil_rate_tC_ha_yr", "first"),
        country_dta_inputs=("iso3", "nunique"),
        nr_cells=("natural_regeneration_cells_selected", "sum"),
        plantation_cells=("plantation_cells_selected", "sum"),
        method_switches=("method_switches_vs_reference", "sum"),
        available_supply_discounted_tco2=("available_discounted_tco2", "sum"),
    )
    .reset_index()
)
display(sensitivity_summary)

sensitivity_curves = pd.read_csv(sensitivity_curves_path)
global_sensitivity_curves = (
    sensitivity_curves.groupby(["soil_scenario", "price_usd2015_per_tco2"], as_index=False)
    ["cumulative_supply_discounted_tco2"].sum()
)
fig, ax = plt.subplots(figsize=(8, 5))
for scenario, rows in global_sensitivity_curves.groupby("soil_scenario"):
    rows = rows.sort_values("price_usd2015_per_tco2")
    ax.plot(
        rows["price_usd2015_per_tco2"],
        rows["cumulative_supply_discounted_tco2"] / 1e6,
        label=scenario,
    )
ax.set_xlabel("Price threshold (2015 US$/tCO₂)")
ax.set_ylabel("Cumulative modelled supply (MtCO₂, discounted over 30 years)")
ax.set_title("All-country supply under alternative natural-regeneration soil rates")
ax.legend(title="Soil scenario")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

soil_sensitivity_coverage = pd.read_csv(sensitivity_coverage_path)
target_sensitivity_summary = (
    soil_sensitivity_coverage.groupby(["soil_scenario", "status"], dropna=False)
    .agg(
        positive_loss_entities=("iso3", "nunique"),
        reported_loss_capacity_tco2_yr=("reported_loss_capacity_tco2_yr", "sum"),
    )
    .reset_index()
)
display(target_sensitivity_summary)

## 5. Read the target-cost results alongside the 2% allocation

The all-country files show the modelled costs and supply for every archive input. Target-cost fields appear in the separate positive-loss coverage table only after the ISO3 join. The 2% GCW allocation remains an annual hypothetical amount in 2015 US dollars; target costs are derived from the 30-year restoration calculation. Their comparison is a scenario comparison, not an observed budget or financing gap.

In [ ]:
comparison_path = VALIDATION / "finance_action_cost_comparison.json"
if comparison_path.exists():
    comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
    display(pd.Series(comparison, name="value").to_frame())
else:
    print("The finance comparison summary is not present yet.")

print(
    "All-country Busch DTA processing complete:",
    busch_summary["processing_scope_complete"],
    "| published country-curve comparison performed:",
    busch_summary["published_curve_comparison_performed"],
)